<a href="https://colab.research.google.com/github/TamaraCucumides/MDS3020/blob/main/Clase3_Practico_EDA_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Práctico Clase 3 — EDA profesional potenciado por IA

En este notebook vamos a practicar un flujo de **análisis descriptivo moderno**, usando herramientas más allá de `pandas` tradicional:

- Validación rápida de datos (`pandera`)
- Exploración automatizada (`ydata-profiling`)
- Agregaciones expresivas y eficientes (`polars`, `duckdb`)
- Visualización interactiva (`plotly.express`)

> En **Google Colab**, aprovecha el **autocompletado** y las opciones de “Generate code / Explain code” para acelerar tu flujo.



## 0. Preparación del entorno

Ejecuta esta celda una vez al inicio.  
En Colab instalará las librerías necesarias.


In [ ]:
!pip install -q pandas polars duckdb pandera ydata-profiling plotly

In [ ]:

# En Colab: instala dependencias (toma unos segundos)
# !pip install -q pandas polars duckdb pandera ydata-profiling plotly

import pandas as pd
import polars as pl
import duckdb
import pandera.pandas as pa
from pandera import Column, Check
from ydata_profiling import ProfileReport
import plotly.express as px

print("Versión pandas:", pd.__version__)
print("Versión polars:", pl.__version__)
print("Versión duckdb:", duckdb.__version__)



## 1. Cargar el dataset

Usaremos el clásico dataset de **propinas en un restaurante** (`tips`), pero analizándolo con herramientas “nivel pro”.


In [ ]:
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
df = pd.read_csv(url)
df.head()



## 2. Validación rápida con `pandera` (fail fast)

Antes de hacer EDA, un analista profesional **valida** que los datos tengan sentido:

- Valores positivos donde corresponde
- Categorías dentro de un conjunto esperado
- Tipos de datos coherentes

Aquí definimos un esquema simple y validamos el `DataFrame`.

Pueden considerar esto como un paso intermedio de validacion de integridad de datos cuando estos no vienen de una DB relacional.


In [ ]:

schema = pa.DataFrameSchema({
    "total_bill": Column(float, Check.gt(0)),
    "tip": Column(float, Check.ge(0)),
    "sex": Column(object, Check.isin(["Male", "Female"])),
    "smoker": Column(object, Check.isin(["Yes", "No"])),
    "day": Column(object, Check.isin(["Thur","Fri", "Sat", "Sun"])),
    "time": Column(object, Check.isin(["Lunch", "Dinner"])),
    "size": Column(int, Check.ge(1)),
})

validated_df = schema.validate(df)
validated_df.head()


In [ ]:
validated_df


## 3. EDA automático con `ydata-profiling`

Esta librería genera un **reporte exploratorio completo**:

- Descriptivos básicos
- Distribuciones
- Valores faltantes
- Correlaciones
- Alertas de calidad de datos


In [ ]:

profile = ProfileReport(validated_df, title="Reporte EDA automático - Tips", explorative=True)
profile.to_notebook_iframe()

In [ ]:

profile = ProfileReport(validated_df, title="Reporte EDA automático - Tips", explorative=True)
profile.to_notebook_iframe()



## 4. Agregaciones modernas con `polars` (Lazy API)

`polars` permite escribir análisis declarativos, eficientes y muy legibles.

Aquí vamos a:

- Convertir el DataFrame a Polars
- Calcular la tasa de propina (`tip_rate`)
- Resumir por día de la semana


In [ ]:

pl_df = pl.from_pandas(validated_df)

resumen_dia = (
    pl_df.lazy()
         .with_columns((pl.col("tip") / pl.col("total_bill")).alias("tip_rate"))
         .group_by("day") # El error que hubo en la clase es que la sintaxis es group_by, y había puesto groupby
         .agg([
             pl.count().alias("n"),
             pl.col("total_bill").mean().alias("mean_bill"),
             pl.col("tip").mean().alias("mean_tip"),
             pl.col("tip_rate").mean().alias("mean_tip_rate")
         ])
         .sort("mean_tip_rate", descending=True)
         .collect()
)

resumen_dia



## 5. Consultas SQL rápidas con `duckdb`

En muchos equipos, el analista debe **hablar SQL**.  
`duckdb` permite consultar un DataFrame como si fuera una tabla de base de datos, sin salir de Python.


In [ ]:

duckdb.sql("CREATE OR REPLACE TABLE tips AS SELECT * FROM df")

duckdb.sql("""
    SELECT
        day,
        AVG(tip/total_bill) AS mean_tip_rate,
        COUNT(*) AS n
    FROM tips
    GROUP BY day
    ORDER BY mean_tip_rate DESC
""").df()



## 6. Visualización interactiva con `plotly.express`

Vamos a construir algunas visualizaciones típicas de EDA, pero con interactividad:

- Relación entre cuenta total y propina
- Distribución de la tasa de propina por día


In [ ]:

df_viz = validated_df.copy()
df_viz["tip_rate"] = df_viz["tip"] / df_viz["total_bill"]

fig_scatter = px.scatter(
    df_viz,
    x="total_bill",
    y="tip",
    size="size",
    color="day",
    hover_data=["sex", "time"],
    title="Relación entre total_bill y tip por día"
)
fig_scatter.show()


In [ ]:

fig_box = px.box(
    df_viz,
    x="day",
    y="tip_rate",
    points="all",
    title="Distribución de la tasa de propina por día"
)
fig_box.show()




### Ejercicio 1 — Segmentación por fumadores

1. Usando `polars` o `duckdb`, calcula la **tasa media de propina (`tip_rate`)** para fumadores vs no fumadores.
2. ¿En qué grupo es mayor la tasa media de propina?
3. Escribe una posible explicación (pueden ser más de una).


In [ ]:

# TODO: completa este análisis con polars
(
    pl_df.lazy()
        .with_columns((pl.col("tip") / pl.col("total_bill")).alias("tip_rate"))
        # .groupby("smoker")
        # .agg([...])
        # .collect()
)



### Ejercicio 2 — Mini–insight visual

1. Construye una visualización con `plotly.express` que te ayude a responder alguna pregunta de negocio, por ejemplo:
   - “¿Qué tamaño de mesa deja más propina en proporción a la cuenta?”  
   - “¿Almuerzo vs cena cambia la propina?”  
2. Escribe una frase de insight y una recomendación.


In [ ]:

# TODO: crea tu propia visualización con plotly.express
# fig = px.bar(df_viz, x=..., y=..., color=...)
# fig.show()



---

### Cierre

En este práctico viste cómo:

- Validar datos rápidamente con `pandera`  
- Obtener una visión global del dataset con `ydata-profiling`  
- Hacer agregaciones expresivas con `polars` y `duckdb`  
- Crear visualizaciones interactivas con `plotly`  

> “Automatiza lo mecánico, pero nunca el pensamiento crítico.”
